In [0]:
from pyspark.sql.functions import col

#Data Ingestion

order_df = spark.read.csv("/Volumes/workspace/default/superstore/data/bronze/SuperStoreOrders.csv", header = True, inferSchema=False, multiLine=True, quote='"', escape='"')

#check duplicates
order_df.groupBy(["order_id","product_id","sub_category","quantity","shipping_cost"]).count().filter("count > 1").show() 

#check for nulls
order_df.filter(" OR ".join([f"`{c}` IS NULL" for c in order_df.columns])).show()

display(order_df)

In [0]:
from pyspark.sql.functions import to_date, coalesce, expr, regexp_replace

cleaned_order_df = order_df.withColumn(
    "order_date",
    coalesce(
        expr("try_to_timestamp(order_date, 'M/d/yyyy')"),
        expr("try_to_timestamp(order_date, 'dd-MM-yyyy')")
    ).cast("date")
).withColumn(
    "ship_date",
    coalesce(
        expr("try_to_timestamp(ship_date, 'M/d/yyyy')"),
        expr("try_to_timestamp(ship_date, 'dd-MM-yyyy')")
    ).cast("date")
).withColumn("sales",regexp_replace(col("sales"), ",", "").cast("double"))
display(cleaned_order_df)

In [0]:
#writing to the silver layer

cleaned_order_df.write.mode("overwrite").parquet("/Volumes/workspace/default/superstore/data/silver/orders_cleaned")
